In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

def load_data(path):
    # Ensure you handle the tab-separation for the SMS dataset
    return pd.read_csv(path, sep='\t', names=['label', 'message'])

def prepare_data(df):
    # split and save
    train, rem = train_test_split(df, train_size=0.7, random_state=42)
    val, test = train_test_split(rem, test_size=0.5, random_state=42)
    train.to_csv('train.csv', index=False)
    val.to_csv('validation.csv', index=False)
    test.to_csv('test.csv', index=False)

In [2]:
import pandas as pd
import requests
import zipfile
import io

# Direct link to the SMS Spam Collection ZIP from UCI
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

# Download and extract
r = requests.get(url)
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    with z.open('SMSSpamCollection') as f:
        df = pd.read_csv(f, sep='\t', names=['label', 'message'])

print(df.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

def load_data(file_path):
    # Assuming tab-separated based on the UCI dataset link
    df = pd.read_csv(file_path, sep='\t', names=['label', 'message'])
    return df

def preprocess_data(df):
    # Convert labels to binary: ham=0, spam=1
    df['label'] = df['label'].map({'ham': 0, 'spam': 1})
    # Basic text cleaning: lowercase
    df['message'] = df['message'].str.lower()
    return df

def split_and_store_data(df):
    # Split: 70% Train, 15% Val, 15% Test
    train, rem = train_test_split(df, train_size=0.7, random_state=42)
    val, test = train_test_split(rem, test_size=0.5, random_state=42)

    train.to_csv('train.csv', index=False)
    val.to_csv('validation.csv', index=False)
    test.to_csv('test.csv', index=False)
    print("Files stored: train.csv, validation.csv, test.csv")

In [7]:
def load_data(url):
    r = requests.get(url)
    # Use io.BytesIO so zipfile can read the raw bytes from the request
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        # We explicitly open ONLY the data file
        with z.open('SMSSpamCollection') as f:
            # Important: We read from 'f' (the specific file), not the whole zip
            df = pd.read_csv(f, sep='\t', names=['label', 'message'])
    return df

In [9]:
# Reset the URL variable to the string
dataset_url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

# Now call the function with the correct variable name
raw_data = load_data(dataset_url)

# Proceed with the rest
cleaned_data = preprocess_data(raw_data)
split_and_store_data(cleaned_data)

Files stored: train.csv, validation.csv, test.csv


In [12]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# --- Define Required Functions ---

def fit_model(model, X_train, y_train):
    """Fits a model on training data."""
    return model.fit(X_train, y_train)

def score_model(model, X_data):
    """Returns predictions for a given model and dataset."""
    return model.predict(X_data)

def evaluate_predictions(y_true, y_pred, set_name="Data"):
    """Prints evaluation metrics."""
    acc = accuracy_score(y_true, y_pred)
    print(f"--- Evaluation on {set_name} ---")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_true, y_pred))
    return acc

def validate_model(model, X_train, y_train, X_val, y_val):
    """Performs the full validation loop required by the prompt."""
    # i. fit on train
    fit_model(model, X_train, y_train)
    # ii. & iii. score and evaluate on train and validation
    train_preds = score_model(model, X_train)
    val_preds = score_model(model, X_val)

    print("Training Performance:")
    evaluate_predictions(y_train, train_preds, "Train")
    print("\nValidation Performance:")
    val_acc = evaluate_predictions(y_val, val_preds, "Validation")
    return val_acc

In [13]:
# 1. Load Data
train_df = pd.read_csv('train.csv')
val_df = pd.read_csv('validation.csv')
test_df = pd.read_csv('test.csv')

# 2. Vectorize Text
tfidf = TfidfVectorizer(stop_words='english')
X_train = tfidf.fit_transform(train_df['message'])
X_val = tfidf.transform(val_df['message'])
X_test = tfidf.transform(test_df['message'])

y_train = train_df['label']
y_val = val_df['label']
y_test = test_df['label']

# 3. Score three benchmark models
benchmark_models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(),
    "SVM": SVC(kernel='linear')
}

best_model = None
best_acc = 0

for name, model in benchmark_models.items():
    print(f"\n{'='*30}\nBenchmarking: {name}\n{'='*30}")
    current_val_acc = validate_model(model, X_train, y_train, X_val, y_val)

    if current_val_acc > best_acc:
        best_acc = current_val_acc
        best_model = model

# 4. Final step: select the best one and score on test data
print(f"\nFinal Selection: The best model is {type(best_model).__name__}")
test_preds = score_model(best_model, X_test)
evaluate_predictions(y_test, test_preds, "FINAL TEST SET")


Benchmarking: Naive Bayes
Training Performance:
--- Evaluation on Train ---
Accuracy: 0.9803
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3377
           1       1.00      0.85      0.92       523

    accuracy                           0.98      3900
   macro avg       0.99      0.93      0.95      3900
weighted avg       0.98      0.98      0.98      3900


Validation Performance:
--- Evaluation on Validation ---
Accuracy: 0.9713
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       724
           1       1.00      0.79      0.88       112

    accuracy                           0.97       836
   macro avg       0.98      0.89      0.93       836
weighted avg       0.97      0.97      0.97       836


Benchmarking: Logistic Regression
Training Performance:
--- Evaluation on Train ---
Accuracy: 0.9636
              precision    recall  f1-score   support

           0       0

0.9916267942583732